<a href="https://colab.research.google.com/github/rahil161190/Movie_SentimentAnalysis_FineTuning_Bert/blob/main/ImdbMovie_Fine_Tuning_Bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install tensorflow==2.16.1
%pip install transformers

# Main goal of this project is to fine-tune a pre-trained BERT model for sequence classification, specifically for sentiment analysis, using the IMDB dataset

In [2]:
from transformers import BertTokenizer

 BERT model itself doesn't "assign" the IDs on the fly; it uses a fixed vocabulary that was learned during its extensive pre-training on a massive amount of text data. The tokenizer's job is to efficiently map new text to this existing vocabulary and its associated IDs

BERT uses a technique called WordPiece tokenization. Here's how it works when it encounters a "new" word:

Subword Segmentation: Instead of only storing whole words, the tokenizer's vocabulary also contains common subword units (prefixes, suffixes, roots, etc.). When it sees a word not in its full-word vocabulary, it tries to break that word down into smaller subword units that are in its vocabulary. For example, if 'unbelievable' isn't in the vocabulary, it might break it into 'un', '##believe', and '##able'. The '##' indicates that it's a continuation of a previous token.

[UNK] Token for Truly Unknown Parts: If a part of a word (or a whole word) cannot be broken down into any known subword units from its vocabulary, the tokenizer will replace that part with a special token called [UNK] (for 'unknown'). This ensures that the model always receives a numerical ID for every part of the input, even if it's an unfamiliar word.

This approach has two main benefits:

Handles Out-of-Vocabulary (OOV) words: The model doesn't just crash or ignore new words; it can still process them by understanding their constituent parts.
Manages Vocabulary Size: Instead of needing an impossibly large vocabulary for every possible word, a more compact vocabulary of common words and subword units can cover a vast range of language.

In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
words = ['parachute','paraglide','scubadiving','scubadiver','scubadive']

In [8]:
for i in words:
  tok_word = tokenizer.tokenize(i)
  print(f'{i} tokenized to {tok_word}')

parachute tokenized to ['parachute']
paraglide tokenized to ['para', '##gli', '##de']
scubadiving tokenized to ['scuba', '##di', '##ving']
scubadiver tokenized to ['scuba', '##di', '##ver']
scubadive tokenized to ['scuba', '##di', '##ve']


In [13]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb")
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [14]:
def tokenize_function(examples):
  return tokenizer(examples['text'],truncation = True,padding = 'max_length')

In [15]:
tokenized_dataset = ds.map(tokenize_function,batched = True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

We need to import the BertForSequenceClassification class from the transformers library before we can use it

In [17]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained('bert-base-uncased',num_labels = 2)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="sentiment_model",
    # evaluation_strategy="epoch",  # Temporarily removed to fix TypeError, consider updating transformers to re-enable.
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
)

In [26]:
from transformers import Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # For binary classification
    return {"accuracy": accuracy_score(labels, predictions)}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'].shuffle(seed =42).select(range(2000)),
    eval_dataset=tokenized_dataset['test'].shuffle(seed =42).select(range(500)), # Added a comma here
    compute_metrics=compute_metrics,
)

In [27]:
trainer.train()

Step,Training Loss
10,0.696182
20,0.681028
30,0.649569
40,0.630235
50,0.466252
60,0.397359
70,0.505577
80,0.319436
90,0.333331
100,0.288648


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.3849572777748108, metrics={'train_runtime': 216.3349, 'train_samples_per_second': 9.245, 'train_steps_per_second': 1.156, 'total_flos': 526222110720000.0, 'train_loss': 0.3849572777748108, 'epoch': 1.0})

In [28]:
trainer.evaluate()

{'eval_loss': 0.28430256247520447,
 'eval_accuracy': 0.892,
 'eval_runtime': 18.0392,
 'eval_samples_per_second': 27.717,
 'eval_steps_per_second': 3.492,
 'epoch': 1.0}

In [34]:
import pandas as pd

small_eval_dataset = tokenized_dataset['test'].shuffle(seed=42).select(range(500))

predictions_output = trainer.predict(small_eval_dataset)

raw_logits = predictions_output.predictions
true_labels = predictions_output.label_ids
predicted_labels = np.argmax(raw_logits, axis=1)

from scipy.special import softmax
probabilities = softmax(raw_logits, axis=1)
confidence_scores = np.max(probabilities, axis=1)

results_df = pd.DataFrame({
    "Review_Text": small_eval_dataset["text"],
    "True_Label": true_labels,
    "Predicted_Label": predicted_labels,
    "Confidence": confidence_scores
})

In [35]:
results_df

,Review_Text,True_Label,Predicted_Label,Confidence
0,<br /><br />When I unsuspectedly rented A Thou...,1,1,0.966106
1,This is the latest entry in the long series of...,1,1,0.698041
2,This movie was so frustrating. Everything seem...,0,0,0.960318
3,"I was truly and wonderfully surprised at ""O' B...",1,1,0.954532
4,This movie spends most of its time preaching t...,0,0,0.949484
...,...,...,...,...
495,I'm not going to say too much as this movie is...,0,0,0.962481
496,Definitely the worst movie I have ever seen in...,0,0,0.963172
497,"Wow, there are no words to describe how bad th...",0,0,0.964510
498,Gritty drama? Emotionally powerful? Blah! The ...,0,0,0.956752
